# nevla-vln — Perception model benchmark (Colab)

**Goal:** pick the SAM2 + open-vocab detector variants that fit the **4050's 6 GB
VRAM at runtime alongside the Unity sim**, and sanity-check that the challenge
object vocabulary is detected.

Run on Colab (GPU runtime) **in parallel** with the Docker bring-up on the 4050.
Self-contained — no repo/sim needed. Output = which models to configure in
SysNav's detection / semantic_mapping on the 4050.

> Runtime → Change runtime type → **GPU** (T4 is fine for sizing).

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
!pip -q install "ultralytics>=8.3.0"

In [ ]:
# Target budget: the 4050 has 6 GB. Reserve for the Unity sim + CUDA/ROS overhead.
TOTAL_VRAM_GB  = 6.0
SIM_RESERVE_GB = 2.0      # Unity rendering on the same GPU
OVERHEAD_GB    = 0.8      # CUDA context + ROS
PERCEPTION_BUDGET_GB = round(TOTAL_VRAM_GB - SIM_RESERVE_GB - OVERHEAD_GB, 1)
IMG_SIZES = [640, 960, 1280]

# Challenge object vocabulary (mined from the 75 questions; ~ challenge_classes.yaml)
CHALLENGE_CLASSES = [
 'potted plant','plant','table','coffee table','dining table','bedside table',
 'tea table','dressing table','tv','tv cabinet','cabinet','file cabinet','pillow',
 'picture','painting','photo','vase','chair','sofa','couch','window','door',
 'door frame','wall lamp','lamp','ceiling lamp','lantern','stool','bed','bench',
 'flower','bowl','clock','fireplace','shelf','bookcase','book','column',
 'folding screen','projector screen','screen','kettle','nightstand','curtain',
 'phone','tv remote','cup','paper cup','whiteboard','guitar','hookah','tray',
 'elephant figurine','horse figurine','knife rack','refrigerator','speaker',
 'magazine','ottoman','decoration','sphere decoration','fan decoration',
 'stone decoration','jar','wardrobe door','candle holder','microwave',
 'kitchen counter','soccer ball','mirror','stairs','water cooler','box','folder',
 'exit sign','beer bottle','easel','computer monitor','sushi','floor',
]
print(len(CHALLENGE_CLASSES), 'classes | perception VRAM budget =', PERCEPTION_BUDGET_GB, 'GB')

In [ ]:
import time, gc, numpy as np, pandas as pd, torch
def reset():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_gb():
    return torch.cuda.max_memory_allocated()/1e9
def synth(h, w):
    return np.random.randint(0, 255, (h, w, 3), dtype=np.uint8)
def bench(fn, n=8, warmup=3):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(n): fn()
    torch.cuda.synchronize()
    return (time.time()-t0)/n*1000.0   # ms / inference

## 1. Open-vocab detector (YOLO-World) — VRAM + latency
Stands in for SysNav's open-vocab detector (same role). Prompted with the 80
challenge classes, swept over input sizes.

In [ ]:
from ultralytics import YOLO
DETECTORS = ['yolov8s-worldv2.pt','yolov8m-worldv2.pt','yolov8l-worldv2.pt','yolov8x-worldv2.pt']
rows = []
for name in DETECTORS:
    for imgsz in IMG_SIZES:
        try:
            reset()
            m = YOLO(name); m.to('cuda'); m.set_classes(CHALLENGE_CLASSES)
            img = synth(imgsz, imgsz)
            lat = bench(lambda: m.predict(img, imgsz=imgsz, device=0, verbose=False))
            rows.append([name, imgsz, round(peak_gb(),2), round(lat,1)])
            del m; reset()
        except Exception as e:
            rows.append([name, imgsz, None, f'ERR {e}'])
det_df = pd.DataFrame(rows, columns=['detector','imgsz','peak_VRAM_GB','latency_ms'])
print(det_df.to_string(index=False))

## 2. SAM2 — VRAM + latency (detection-prompted, SysNav-style)
Benchmarked with box prompts (far lighter than segment-everything).

In [ ]:
from ultralytics import SAM
SAMS = ['sam2.1_t.pt','sam2.1_s.pt','sam2.1_b.pt','sam2.1_l.pt']
rows = []
for name in SAMS:
    try:
        reset()
        s = SAM(name); s.to('cuda')
        img = synth(1024, 1024)
        boxes = [[100,100,400,400],[500,300,820,720]]
        lat = bench(lambda: s.predict(img, bboxes=boxes, device=0, verbose=False), n=5, warmup=2)
        rows.append([name, round(peak_gb(),2), round(lat,1)])
        del s; reset()
    except Exception as e:
        rows.append([name, None, f'ERR {e}'])
sam_df = pd.DataFrame(rows, columns=['sam2','peak_VRAM_GB','latency_ms'])
print(sam_df.to_string(index=False))

## 3. Combined (detector + SAM2 resident) vs the 6 GB budget
Both models loaded + run once → realistic pipeline peak. Verdict against the
perception budget (6 GB − sim − overhead).

In [ ]:
def combined_peak(det_name, sam_name, imgsz=960):
    reset()
    det = YOLO(det_name); det.to('cuda'); det.set_classes(CHALLENGE_CLASSES)
    sam = SAM(sam_name); sam.to('cuda')
    img = synth(imgsz, imgsz)
    det.predict(img, imgsz=imgsz, device=0, verbose=False)
    sam.predict(img, bboxes=[[100,100,400,400]], device=0, verbose=False)
    p = peak_gb(); del det, sam; reset(); return round(p,2)

COMBOS = [
 ('yolov8s-worldv2.pt','sam2.1_t.pt'),
 ('yolov8m-worldv2.pt','sam2.1_s.pt'),
 ('yolov8l-worldv2.pt','sam2.1_b.pt'),
 ('yolov8x-worldv2.pt','sam2.1_l.pt'),
]
print(f'perception budget = {PERCEPTION_BUDGET_GB} GB  (6 - sim {SIM_RESERVE_GB} - overhead {OVERHEAD_GB})\n')
for d, s in COMBOS:
    try:
        v = combined_peak(d, s)
        verdict = 'FITS' if v <= PERCEPTION_BUDGET_GB else ('TIGHT' if v <= PERCEPTION_BUDGET_GB+1 else 'OVER')
        print(f'{d:22s} + {s:12s}  peak={v:5.2f} GB  -> {verdict}')
    except Exception as e:
        print(f'{d} + {s}: ERR {e}')

## 4. (Optional) detection-quality check on real frames
Upload a few real indoor frames (sim screenshots once the 4050 is up, or any
room photos) to confirm the open-vocab prompts fire on challenge objects.
VRAM/latency above is content-independent; this checks quality.

In [ ]:
from google.colab import files
import cv2, matplotlib.pyplot as plt
from ultralytics import YOLO
up = files.upload()
det = YOLO('yolov8x-worldv2.pt'); det.to('cuda'); det.set_classes(CHALLENGE_CLASSES)
for fn in up:
    img = cv2.imread(fn)
    r = det.predict(img, imgsz=1280, conf=0.03, device=0, verbose=False)[0]
    labels = [CHALLENGE_CLASSES[int(c)] for c in r.boxes.cls.tolist()] if r.boxes is not None else []
    print(fn, '->', len(labels), 'dets:', sorted(set(labels)))
    plt.figure(figsize=(11,7)); plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title(fn); plt.show()

## How to use these results (on the 4050)
- Pick the **largest combo marked FITS/TIGHT** — that detector + SAM2 size is what
  to configure in SysNav (`detection_node` / `semantic_mapping`), with
  `object_file` pointed at `challenge_classes.yaml`.
- If even the smallest is **OVER**: lower `imgsz`, run perception at a reduced
  rate, or don't run the sim at full resolution at the same time.
- Latency feeds the 10-min/question budget: (detector + SAM2) per frame × frames
  explored must stay well under it.

Notes: YOLO-World stands in for SysNav's open-vocab detector (YOLOe sizes are
comparable). SAM2 is benchmarked detection-prompted (SysNav-style), much lighter
than segment-everything.